# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Load the supplied dataset (works when this notebook and CSV are in the same folder,
# and also with the original course folder structure).
csv_candidates = [Path("co2_emissions.csv"), Path("../data/co2_emissions.csv")]
csv_file = next((p for p in csv_candidates if p.exists()), None)
if csv_file is None:
    raise FileNotFoundError("Place co2_emissions.csv beside this notebook or in ../data/.")

df = pd.read_csv(csv_file)
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
display(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022


,Country,Region,Year,CO2_Mt,CO2_per_capita
0,United States,North America,2000,5857.6,1.32
1,United States,North America,2001,5724.0,1.26
2,United States,North America,2002,5652.8,1.11
3,United States,North America,2003,5592.8,1.29
4,United States,North America,2004,5743.2,1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [3]:
# Task — Multi-series line with highlight
# Show every Asian country in light grey, with China highlighted in colour.

asia = df[df["Region"].eq("Asia")].copy()
highlight_country = "China"

fig = go.Figure()

# Context lines: every Asian country except the highlighted country.
for country, country_df in asia.groupby("Country", sort=True):
    if country == highlight_country:
        continue
    country_df = country_df.sort_values("Year")
    fig.add_trace(go.Scatter(
        x=country_df["Year"],
        y=country_df["CO2_Mt"],
        mode="lines",
        name=country,
        line=dict(color="#EEEEEE", width=1.2),
        hovertemplate=f"{country}<br>Year: %{{x}}<br>CO₂: %{{y:,.1f}} Mt<extra></extra>",
        showlegend=False
    ))

# Highlight China.
china = asia[asia["Country"].eq(highlight_country)].sort_values("Year")
if china.empty:
    raise ValueError(f"{highlight_country} is not present in the Asian-country data.")

fig.add_trace(go.Scatter(
    x=china["Year"],
    y=china["CO2_Mt"],
    mode="lines",
    name=highlight_country,
    line=dict(color="#D95F02", width=3.5),
    hovertemplate="China<br>Year: %{x}<br>CO₂: %{y:,.1f} Mt<extra></extra>",
    showlegend=False
))

# Direct label at the end of the highlighted line (no legend needed).
last = china.iloc[-1]
fig.add_annotation(
    x=last["Year"],
    y=last["CO2_Mt"],
    text="China",
    showarrow=False,
    xshift=30,
    yshift=0,
    font=dict(size=13, color="#D95F02", family="Arial")
)

fig.update_layout(
    title=dict(
        text="China’s CO₂ emissions rose sharply, adding 8,896 Mt from 2000 to 2022",
        x=0,
        xanchor="left"
    ),
    xaxis_title="Year",
    yaxis_title="CO₂ emissions (Mt)",
    template="plotly_white",
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=65, r=95, t=85, b=55),
    font=dict(family="Arial", size=12)
)
fig.update_xaxes(showline=False, showgrid=False, zeroline=False, dtick=5)
fig.update_yaxes(showline=False, showgrid=True, gridcolor="#F0F0F0", zeroline=False)
fig.show()
